### **DE_NB_LGL_REG_CREATE_TABLES**
___
###### **Purpose:** Creation of Legal and Regulatory Reporting Tables

###### **Sources (input tables):** 

###### **Sink Tables (outputs)**: 
- dp_lgl_reg_dim_rpg_ent
- dp_lgl_reg_dim_rpg_idt
- dp_lgl_reg_acg_csl_ldgr
- dp_lgl_reg_fact_scrts
- dp_lgl_reg_prty
- dp_lgl_reg_rlt_instrmnt_prty
- dp_lgl_reg_dim_ent_hier
- dp_lgl_reg_int_ent_prm
- dp_lgl_reg_int_intra_grp
- dp_lgl_reg_acg_inst_csl_ldgr
- dp_lgl_reg_int_acg_csl_adj
- dp_lgl_reg_acg_csl_rec
- dp_lgl_reg_dim_drvt_inst
- dp_lgl_reg_acg_csl_drvt
- dp_lgl_reg_instrmnt_ori
- dp_lgl_reg_cdelst
- dp_lgl_reg_dpm_param_datapoint
- dp_lgl_reg_dpm_param_fltr
- dp_lgl_reg_dpm_param_ctlg
- dp_lgl_reg_dpm_param_src_tbl
- dp_lgl_reg_dpm_param_iss_ctlg
- dp_lgl_reg_dpm_ref_src_datapoint_typ
- dp_lgl_reg_dpm_ref_tmplt_csl_perim
- dp_lgl_reg_dpm_ref_tmplt_ctry
- dp_lgl_reg_dpm_param_vld
- dp_lgl_reg_dpm_param_load
- dp_lgl_reg_dpm_exec
- dp_lgl_reg_dpm_exec_step
- dp_lgl_reg_dpm_iss_log
- dp_lgl_reg_dpm_result_var_val
- dp_lgl_reg_dpm_result_datapoint_val
- dp_lgl_reg_dpm_failed_var_key
- dp_lgl_reg_dpm_lineage_var
- dp_lgl_reg_dpm_lineage_datapoint
- dp_lgl_reg_dpm_datapoint_def
- dp_lgl_reg_dpm_rpt_datapoint

###### **Parameters:**

___
###### **Changelog:**
- 2026-11-17 | Laura Pellicer | DPM engine tables (dp_lgl_reg_dpm_*)
- 2026-11-18 | Laura Pellicer | DPM: val_typ in dp_lgl_reg_dpm_param_ctlg; framework_cd in dp_lgl_reg_dpm_ref_tmplt_csl_perim and dp_lgl_reg_dpm_ref_tmplt_ctry

###### **Technical debt**
- table_example_AB - technical debt - description

###### **Relevant notes**
- ER Studio model: **[Not yet created](link)**.
- Table definition based on mapping: **[Not yet created](link)**.

___


#### **Session Configuration**
___

In [1]:
%%configure -f
{  
"driverMemory": "28g",   
"driverCores": 4,   
"executorMemory": "28g",   
"executorCores": 4,
    "conf": {       
"spark.native.enabled": "true",       
"spark.dynamicAllocation.maxExecutors": "2"
   }
}

StatementMeta(, e12b3b96-b328-4cf6-b68e-45277003611b, -1, Finished, Available, Finished, True)

<div style="font-family:monospace;font-size:13px">
    <p>
        <span style="color:red;font-weight:bold">Warning:</span> When you want to configure the driverMemory, driverCores, executorCores,
        executorMemory, or numExecutors, check the workspace settings and default pool node details from workspace settings -> Default pool
        for workspace -> Pool details, you can use %%configure set customized compute configurations that within the range of Pool level size.
    </p>
    <p>
        Notice that if the session configuration is different from the live session (8 vCores, 56GB memory, 2 executors), the session
        start will fall back to the on-demand session which will take about 3~5 mins to spin up.
    </p>
    <p>The Executor/Driver size buckets on Fabric are: </p>
    <ul>
        <li>A small size bucket: 4 vCores/28 GB memory</li>
        <li>A medium size bucket: 8 vCores/56 GB memory</li>
        <li>A large size bucket: 16 vCores/112 GB memory</li>
        <li>A xlarge size bucket: 32 vCores/224 GB memory</li>
        <li>A xxlarge size bucket: 64 vCores/400 GB memory</li>
    </ul>
    <p>
        We recommend avoiding using Driver or Executor vCore/Memory counts that are not in the size bucket specified above, so be sure
        to specify the <b>Driver and Executor</b> vCore/Memory values in relation to the above size buckets (4vCores/28GB, 8vCores/56GB,
        16vCores/112GB, 32vCores/224GB, or 64vCores/400GB), moreover, the executor size and driver size should be the same.
    </p>
</div>

#### **Imports**
___

In [2]:
from workerFactory import workerfactory
import medallion_framework
from pyspark.sql.utils import AnalysisException

StatementMeta(, 7046c2f2-d03c-415d-aaf3-60a26991c97b, 6, Finished, Available, Finished, False)

<frozen importlib._bootstrap>:283: DeprecationWarning: the load_module() method is deprecated and slated for removal in Python 3.12; use exec_module() instead


#### **Procedures**
___

In [3]:
def create_table(source_table_name, table_type, column_list, reset_list, partition_cols, guids):
    reset_value = next((item["reset"] for item in reset_list if item["name"] == source_table_name), "False")
    reset = reset_value.lower() == "true"

    try:        
        if table_type == "SCD2":
            medallion_framework.create_table_with_control(source_table_name, column_list, drop_if_exists=reset, partition_cols=partition_cols)
        elif table_type == "SCD22":
            medallion_framework.create_table_with_control_SCD22(source_table_name, column_list, drop_if_exists=reset, partition_cols=partition_cols)
        elif table_type == "TABLE":
            medallion_framework.create_table(source_table_name, column_list, drop_if_exists=reset, partition_cols=partition_cols)
        else:
            print(f"{table_type} isn't a valid table reference.")
            return

        print(f"Table created - Name: {source_table_name}; Type: {table_type}; Partitions: {partition_cols}")

        if reset:
            for guid in guids:
                spark.sql(f"DELETE FROM mdfrw_ctrl_processcheckpoint WHERE process_name LIKE '%"+guid+"%'")
                print(f"Checkpoint deleted - guid: {guid}")

    except AnalysisException as e:
        if 'TABLE_OR_VIEW_ALREADY_EXISTS' in str(e): 
            print(f"Table {source_table_name} already exists. Skipping.")

StatementMeta(, 7046c2f2-d03c-415d-aaf3-60a26991c97b, 7, Finished, Available, Finished, False)

### **Table Definition**
___

In [4]:
reset_list = [
    {"name": "dp_lgl_reg_dim_rpg_ent",         "reset": "False"},
    {"name": "dp_lgl_reg_dim_rpg_idt",         "reset": "False"},
    {"name": "dp_lgl_reg_acg_csl_ldgr",        "reset": "False"},
    {"name": "dp_lgl_reg_fact_scrts",          "reset": "False"},
    {"name": "dp_lgl_reg_prty",                "reset": "False"},
    {"name": "dp_lgl_reg_dim_ent_hier",        "reset": "False"},
    {"name": "dp_lgl_reg_rlt_instrmnt_prty",   "reset": "False"},
    {"name": "dp_lgl_reg_int_ent_prm",         "reset": "False"},
    {"name": "dp_lgl_reg_prcss_rules",         "reset": "False"},
    {"name": "dp_lgl_reg_dim_drvt_inst",       "reset": "False"},
    {"name": "dp_lgl_reg_int_intra_grp",       "reset": "False"},
    {"name": "dp_lgl_reg_acg_inst_csl_ldgr",   "reset": "False"},
    {"name": "dp_lgl_reg_int_acg_csl_adj",     "reset": "False"},
    {"name": "dp_lgl_reg_acg_csl_rec",         "reset": "False"},
    {"name": "dp_lgl_reg_acg_csl_drvt",        "reset": "False"},
    {"name": "dp_lgl_reg_instrmnt_ori",        "reset": "False"},
    {"name": "dp_lgl_reg_cdelst",              "reset": "False"},
    {"name": "dp_lgl_reg_sec_hld",             "reset": "False"},
    {"name": "dp_lgl_reg_sec_hld_cur",         "reset": "False"},
    {"name": "dp_lgl_reg_int_acg_msr",         "reset": "True"},
    {"name": "dp_lgl_reg_dpm_param_datapoint",      "reset": "False"},
    {"name": "dp_lgl_reg_dpm_param_fltr",           "reset": "False"},
    {"name": "dp_lgl_reg_dpm_param_ctlg",           "reset": "False"},
    {"name": "dp_lgl_reg_dpm_param_src_tbl",        "reset": "False"},
    {"name": "dp_lgl_reg_dpm_param_iss_ctlg",       "reset": "False"},
    {"name": "dp_lgl_reg_dpm_ref_src_datapoint_typ", "reset": "False"},
    {"name": "dp_lgl_reg_dpm_ref_tmplt_csl_perim",  "reset": "False"},
    {"name": "dp_lgl_reg_dpm_ref_tmplt_ctry",       "reset": "False"},
    {"name": "dp_lgl_reg_dpm_param_vld",            "reset": "False"},
    {"name": "dp_lgl_reg_dpm_param_load",           "reset": "False"},
    {"name": "dp_lgl_reg_dpm_exec",                 "reset": "False"},
    {"name": "dp_lgl_reg_dpm_exec_step",            "reset": "False"},
    {"name": "dp_lgl_reg_dpm_iss_log",              "reset": "False"},
    {"name": "dp_lgl_reg_dpm_result_var_val",       "reset": "False"},
    {"name": "dp_lgl_reg_dpm_result_datapoint_val", "reset": "False"},
    {"name": "dp_lgl_reg_dpm_failed_var_key",       "reset": "False"},
    {"name": "dp_lgl_reg_dpm_lineage_var",          "reset": "False"},
    {"name": "dp_lgl_reg_dpm_lineage_datapoint",    "reset": "False"},
    {"name": "dp_lgl_reg_dpm_datapoint_def",        "reset": "False"},
    {"name": "dp_lgl_reg_dpm_rpt_datapoint",        "reset": "False"}
]

StatementMeta(, 7046c2f2-d03c-415d-aaf3-60a26991c97b, 8, Finished, Available, Finished, False)

#### **Main Tables**
___

##### **dp_lgl_reg_dim_rpg_ent**

In [ ]:
guids = {"fa443aa3-7729-4828-80ea-851c517b9935"}
source_table_name = "dp_lgl_reg_dim_rpg_ent"
                 
columns_list = [
    {"name": "del_flg", "data_type": "int"},
    {"name": "rej_flg", "data_type": "int"},
    {"name": "ent_id", "data_type": "string not null"}
]

partition_cols = []

create_table(source_table_name, 'SCD22', columns_list, reset_list, partition_cols, guids)

StatementMeta(, 31cc9732-219c-4a6a-8461-7120858f1259, 8, Finished, Available, Finished, False)

Table dp_lgl_reg_dim_rpg_ent already exists. Skipping.


##### **dp_lgl_reg_dim_rpg_idt**

In [ ]:
guids = {"12b6c0fd-1bbd-4dad-aeec-1ea8fde67f85"}
source_table_name = "dp_lgl_reg_dim_rpg_idt"
                 
columns_list = [
    {"name": "del_flg", "data_type": "int"},
    {"name": "rej_flg", "data_type": "int"},
    {"name": "acg_idt_idc_id", "data_type": "string not null"},
    {"name": "acg_idt_idc_ln", "data_type": "string"},
    {"name": "idt_tmplt_id", "data_type": "string"},
    {"name": "idt_tmplt_ln", "data_type": "string"}
]

partition_cols = []

create_table(source_table_name, 'SCD22', columns_list, reset_list, partition_cols, guids)

StatementMeta(, 31cc9732-219c-4a6a-8461-7120858f1259, 9, Finished, Available, Finished, False)

Table dp_lgl_reg_dim_rpg_idt already exists. Skipping.


##### **dp_lgl_reg_acg_csl_ldgr**

In [ ]:
guids = {"94710cdb-2fad-467d-af4b-e5153b2e6941"}
source_table_name = "dp_lgl_reg_acg_csl_ldgr"
                 
columns_list = [
    {"name": "prcss_dt", "data_type": "timestamp"},
    {"name": "bal_dt", "data_type": "date"},
    {"name": "ent_id", "data_type": "string not null"},
    {"name": "acg_ntrl_acc_id", "data_type": "string not null"},
    {"name": "acg_msr_id", "data_type": "string not null"},
    {"name": "acg_cfm_pdt_id", "data_type": "string not null"},
    {"name": "acg_fll_acc_id", "data_type": "string not null"},
    {"name": "acg_csl_lvl_cd", "data_type": "string"},
    {"name": "ent_dmstc_cd", "data_type": "string"},
    {"name": "acg_bal_eur_amnt", "data_type": "decimal(38,18)"}
]

partition_cols = ["bal_dt"]

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

StatementMeta(, 31cc9732-219c-4a6a-8461-7120858f1259, 10, Finished, Available, Finished, False)

Table dp_lgl_reg_acg_csl_ldgr already exists. Skipping.


##### **dp_lgl_reg_fact_scrts**

In [ ]:
guids = {"8e6048a5-dbf5-4732-ad1f-0266bb498945"}
source_table_name = "dp_lgl_reg_fact_scrts"
                 
columns_list = [
    {"name": "prcss_dt", "data_type": "timestamp not null"},
    {"name": "yr_nr", "data_type": "string not null"},
    {"name": "yr_mnth_nr", "data_type": "string not null"},
    {"name": "ent_id", "data_type": "string not null"},
    {"name": "rlt_ent_id", "data_type": "string not null"},
    {"name": "ref_dt", "data_type": "date not null"},
    {"name": "isin_prov_id", "data_type": "string"},
    {"name": "isin_id", "data_type": "string not null"},
    {"name": "isin_vax_id", "data_type": "string"},
    {"name": "siet_id", "data_type": "string"},
    {"name": "instrmnt_id", "data_type": "string not null"},
    {"name": "cntrct_ext_id", "data_type": "string"},
    {"name": "prtf_cd", "data_type": "string"},
    {"name": "prtf_mod_typ", "data_type": "string"},
    {"name": "prtf_typ", "data_type": "string"},
    {"name": "ifrs_bs_cptn_ntrl_acc_id", "data_type": "string"},
    {"name": "ifrs_hdg_fv_adj_cptn_ntrl_acc_id", "data_type": "string"},
    {"name": "ifrs_intrst_cptn_ntrl_acc_id", "data_type": "string"},
    {"name": "ifrs_imp_cptn_ntrl_acc_id", "data_type": "string"},
    {"name": "hdg_fv_adj_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "iss_sec_clf_cd", "data_type": "string"},
    {"name": "sec_typ_cd", "data_type": "string"},
    {"name": "iss_dt", "data_type": "date"},
    {"name": "mat_dt", "data_type": "date"},
    {"name": "lst_acqstn_dt", "data_type": "date"},
    {"name": "zero_cpn_flg", "data_type": "int"},
    {"name": "iss_sny_cd", "data_type": "string"},
    {"name": "scrtstn_flg", "data_type": "int"},
    {"name": "scrtstn_trns_flg", "data_type": "int"},
    {"name": "scrtstn_trns_id", "data_type": "string"},
    {"name": "stck_exh_loc_cd", "data_type": "string"},
    {"name": "lstd_sec_flg", "data_type": "int"},
    {"name": "cur_cd", "data_type": "string"},
    {"name": "prty_id", "data_type": "string"},
    {"name": "pdt_id", "data_type": "string"},
    {"name": "crrng_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "nom_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "unt_nom_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "fc_val_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "hld_qty_nr", "data_type": "decimal(38,17)"},
    {"name": "accrd_ntrst", "data_type": "decimal(38,17)"},
    {"name": "imprmnt_amnt", "data_type": "decimal(38,17)"},
    {"name": "eq_prtcptn_imp_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "mkt_val_amnt", "data_type": "decimal(38,17)"},
    {"name": "grss_crryng_amnt", "data_type": "decimal(38,17)"},
    {"name": "fv_typ_cd", "data_type": "string"},
    {"name": "iss_typ", "data_type": "string"},
    {"name": "clltrl_flg", "data_type": "int"},
    {"name": "scrt_typ", "data_type": "string"},
    {"name": "cap_flr_flg", "data_type": "string"},
    {"name": "instrmnt_cla_cd", "data_type": "string"},
    {"name": "prmy_assts_cla_cd", "data_type": "string"},
    {"name": "col_aea_cd", "data_type": "string"},
    {"name": "def_cd", "data_type": "string"},
    {"name": "gen_crp_mntr_strct_cd", "data_type": "string"},
    {"name": "op_bns_mntr_strct_cd", "data_type": "string"},
    {"name": "stry_cd", "data_type": "string"},
    {"name": "stry_typ", "data_type": "string"},
    {"name": "fi_isr_cd", "data_type": "string"},
    {"name": "fdr_cd", "data_type": "string"},
    {"name": "qte_typ", "data_type": "string"},
    {"name": "imprm", "data_type": "string"},
    {"name": "met_val_cd", "data_type": "string"},
    {"name": "crrt_vl_mdl", "data_type": "string"},
    {"name": "fv_opt_mtv_cd", "data_type": "string"},
    {"name": "rdf_end_mtv", "data_type": "string"},
    {"name": "nace_isr_cd", "data_type": "string"},
    {"name": "fv_val_hier_lvl_cd", "data_type": "string"},
    {"name": "npe_cd", "data_type": "string"},
    {"name": "est_prfm", "data_type": "string"},
    {"name": "ste_rdf", "data_type": "string"},
    {"name": "gme_sctr_cd", "data_type": "string"},
    {"name": "inv_sctr_cd", "data_type": "string"},
    {"name": "me_sctr_cd", "data_type": "string"},
    {"name": "stg", "data_type": "string"},
    {"name": "gar_doc_typ", "data_type": "string"},
    {"name": "grntr_typ_id", "data_type": "string"},
    {"name": "sny_typ", "data_type": "string"},
    {"name": "siet_typ", "data_type": "string"},
    {"name": "val_cd", "data_type": "string"},
    {"name": "def_instrmnt", "data_type": "string"},
    {"name": "rdf_strat_dt", "data_type": "date"},
    {"name": "def_start_stts", "data_type": "date"},
    {"name": "rdf_start", "data_type": "date"},
    {"name": "rdf_start_dt", "data_type": "date"},
    {"name": "dat_vle_stg", "data_type": "date"},
    {"name": "fst_vlo_dt", "data_type": "date"},
    {"name": "qte_dt", "data_type": "date"},
    {"name": "def_instrmnt_dt", "data_type": "date"},
    {"name": "def_start_dt", "data_type": "date"},
    {"name": "ste_rdf_dt", "data_type": "date"},
    {"name": "opt_el_dt", "data_type": "date"},
    {"name": "ivs_ent_actvty_start_dt", "data_type": "date"},
    {"name": "cntrct_start_isn_dt", "data_type": "date"},
    {"name": "tnr_mat_dt", "data_type": "string"},
    {"name": "od_pym_dt", "data_type": "date"},
    {"name": "ste_prfm_dt", "data_type": "date"},
    {"name": "fst_cpn_dt", "data_type": "date"},
    {"name": "nxt_pym_dt", "data_type": "date"},
    {"name": "rtg_dt", "data_type": "date"},
    {"name": "rlp_dt", "data_type": "date"},
    {"name": "sttlmnt_dt", "data_type": "date"},
    {"name": "cpn_lst_dt", "data_type": "date"},
    {"name": "cptl_vlo_dt", "data_type": "date"},
    {"name": "ntrst_vlo_dt", "data_type": "date"},
    {"name": "cml_ppr_mat_dt", "data_type": "date"},
    {"name": "idae", "data_type": "string"},
    {"name": "cpn_cur", "data_type": "string"},
    {"name": "acg_ent", "data_type": "string"},
    {"name": "bnk_strctr_inl_id", "data_type": "string"},
    {"name": "orig_bnk_strctr_cd", "data_type": "string"},
    {"name": "orig_bnk_strctr_bydate_cd", "data_type": "string"},
    {"name": "domc_bnk_strctr_cd", "data_type": "string"},
    {"name": "domc_bnk_strctr_bydate_cd", "data_type": "string"},
    {"name": "bnk_strctr_cd", "data_type": "string"},
    {"name": "bnk_strctr_bydate_cd", "data_type": "string"},
    {"name": "src_finrep_id", "data_type": "string"},
    {"name": "ent_ctry_lgl_cd", "data_type": "string"},
    {"name": "grnt_prty_id", "data_type": "string"},
    {"name": "isr_sgm_rsk", "data_type": "string"},
    {"name": "acg_sub_ent", "data_type": "string"},
    {"name": "src_tbl_id", "data_type": "string"},
    {"name": "old_pdt_cd", "data_type": "string"},
    {"name": "ctry_cd", "data_type": "string"},
    {"name": "imp_flg", "data_type": "int"},
    {"name": "adj_dt_intrst_acr_flg", "data_type": "int"},
    {"name": "rat_fxd_begin_end_flag", "data_type": "int"},
    {"name": "cap_gns_flg", "data_type": "string"},
    {"name": "embd_deriv_flg", "data_type": "int"},
    {"name": "def_isr", "data_type": "string"},
    {"name": "elimin_flg", "data_type": "int"},
    {"name": "imp_typ", "data_type": "string"},
    {"name": "rdf_flag", "data_type": "string"},
    {"name": "sppi_flag", "data_type": "string"},
    {"name": "mdf_csh_flw_flg", "data_type": "int"},
    {"name": "trgr_low_rsk_flag", "data_type": "string"},
    {"name": "intrst_acr_nm", "data_type": "string"},
    {"name": "blmgd_id_nm", "data_type": "string"},
    {"name": "obg_cat_nm", "data_type": "string"},
    {"name": "placel_inves", "data_type": "string"},
    {"name": "cnv_nm", "data_type": "string"},
    {"name": "div_yld", "data_type": "string"},
    {"name": "isr_nm", "data_type": "string"},
    {"name": "cpn_frq_nm", "data_type": "string"},
    {"name": "rat_fxd_frq", "data_type": "string"},
    {"name": "hqla_nm", "data_type": "string"},
    {"name": "rat_typ", "data_type": "string"},
    {"name": "isr_lei_nm", "data_type": "string"},
    {"name": "mkt_nm", "data_type": "string"},
    {"name": "vlu_mdl", "data_type": "string"},
    {"name": "obser", "data_type": "string"},
    {"name": "dbrs_rtg_nm", "data_type": "string"},
    {"name": "rtg_inl_nm", "data_type": "string"},
    {"name": "rtg_mood_nm", "data_type": "string"},
    {"name": "rtg_sp_nm", "data_type": "string"},
    {"name": "rsk_sctr_nm", "data_type": "string"},
    {"name": "vlu_cd", "data_type": "string"},
    {"name": "amrtz_typ", "data_type": "string"},
    {"name": "yld_cl_mtd", "data_type": "string"},
    {"name": "rat_intrst_var_nm", "data_type": "string"},
    {"name": "acqstn_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "exh_pos_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "npv_acv_cvrg_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "acd_imp_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "prc_intrst_rat_hdg", "data_type": "decimal(38,17)"},
    {"name": "capsoc_amnt", "data_type": "decimal(38,17)"},
    {"name": "iss_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "off_bal_intrst_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "rvsd_ntrst_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "acr_intrst_rph_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "ovrd_intrst_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "mkt_col_scrts_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "nom_rph_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "orn_pos_acqstn_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "pos_prm_dis_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "iss_prc_amnt", "data_type": "decimal(38,17)"},
    {"name": "rfnd_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "rhp_eur_amnt", "data_type": "decimal(38,17)"},  
    {"name": "clbl_ptbl_obg_rfnd_amnt", "data_type": "decimal(38,17)"},
    {"name": "rsv_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "acd_cap_gns_amnt", "data_type": "decimal(38,17)"},
    {"name": "cap_gns_yr_amnt", "data_type": "decimal(38,17)"},
    {"name": "cap_gns_yr_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "acd_cap_gns_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "acd_cap_lss_amnt", "data_type": "decimal(38,17)"},
    {"name": "cap_lss_yr_amnt", "data_type": "decimal(38,17)"},
    {"name": "cap_lss_yr_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "acd_cap_lss_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "imp_rsv_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "unt_acqstn_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "unt_bal_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "var_intrst_rat_mlt", "data_type": "decimal(38,17)"},
    {"name": "ovrd_nr_dys", "data_type": "decimal(38,17)"},
    {"name": "iss_obg_nr", "data_type": "decimal(38,17)"},
    {"name": "od_pym_nr", "data_type": "decimal(38,17)"},
    {"name": "gar_doc_id", "data_type": "string"},
    {"name": "doc_id_ivs", "data_type": "string"},
    {"name": "cpn_frq", "data_type": "decimal(38,17)"},
    {"name": "iss_shr_nr", "data_type": "decimal(38,17)"},
    {"name": "qty_rph", "data_type": "decimal(38,17)"},
    {"name": "dbrs_rtg_typ", "data_type": "string"},
    {"name": "rtg_mood_typ", "data_type": "string"},
    {"name": "rtg_sp_typ", "data_type": "string"},
    {"name": "typ_dpr", "data_type": "string"},
    {"name": "crrt_rat", "data_type": "decimal(38,17)"},
    {"name": "cpn_rat", "data_type": "decimal(38,17)"},
    {"name": "imp_rat", "data_type": "decimal(38,17)"},
    {"name": "intrst_rat", "data_type": "decimal(38,17)"},
    {"name": "imp_lgd_rat", "data_type": "decimal(38,17)"},
    {"name": "ori_nom_cntrct_rat", "data_type": "decimal(38,17)"},
    {"name": "prtcptn_pct_rat", "data_type": "decimal(38,17)"},
    {"name": "yld_rat", "data_type": "decimal(38,17)"},
    {"name": "reprc_rat_cap", "data_type": "decimal(38,17)"},
    {"name": "reprc_rat_flr", "data_type": "decimal(38,17)"},
    {"name": "sprd", "data_type": "decimal(38,17)"},
    {"name": "fnl_yld_rat", "data_type": "decimal(38,17)"},
    {"name": "csl_lvl", "data_type": "string"},
    {"name": "isr_id_typ", "data_type": "string"},
    {"name": "isr_typ", "data_type": "string"},
    {"name": "isr_sub_ent", "data_type": "string"},
    {"name": "ntr_vlu", "data_type": "string"},
    {"name": "acg_ntrl_acc_id", "data_type": "string"},
    {"name": "trd_scrts_flg", "data_type": "int"},
    {"name": "scrts_nm", "data_type": "string"},
    {"name": "rfnd_call_put_typ", "data_type": "string"},
    {"name": "cnvrs_flg", "data_type": "int"},
    {"name": "rat_ref_nm", "data_type": "string"},
    {"name": "qte_expss_typ", "data_type": "string"},
    {"name": "idx_nm", "data_type": "string"},
    {"name": "lcl_scrts_cd", "data_type": "string"},
    {"name": "inl_scrts_cd", "data_type": "string"},
    {"name": "cusip_cd", "data_type": "string"},
    {"name": "shrt_scrts_nm", "data_type": "string"},
    {"name": "lst_cpn_dur_per", "data_type": "string"},
    {"name": "nxt_reprc_dt", "data_type": "date"},
    {"name": "ret_scrts_flg", "data_type": "int"},
    {"name": "ngt_scrts_idx_cd", "data_type": "string"},
    {"name": "acg_intrst_ntrl_acc_id", "data_type": "string"},
    {"name": "acg_fv_pstv_ntrl_acc_id", "data_type": "string"},
    {"name": "acg_fv_neg_ntrl_acc_id", "data_type": "string"},
    {"name": "acg_imp_ntrl_acc_id", "data_type": "string"},
    {"name": "acg_rph_ntrl_acc_id", "data_type": "string"},
    {"name": "mrkt_rat_cd", "data_type": "string"},
    {"name": "sny_scrts_od", "data_type": "string"},
    {"name": "acg_exh_pos_ntrl_acc_id", "data_type": "string"},
    {"name": "acg_off_bal_intrst_ntrl_acc_id", "data_type": "string"},
    {"name": "acg_rvsd_intrst_ntrl_acc_id", "data_type": "string"},
    {"name": "acg_rsv_imp_ntrl_acc_id", "data_type": "string"},
    {"name": "acg_acd_fv_neg_ntrl_acc_id", "data_type": "string"},
    {"name": "acg_acd_fv_pstv_ntrl_acc_id", "data_type": "string"},
    {"name": "flg_stg_low_rsk", "data_type": "int"},
    {"name": "cat_scrts_desc", "data_type": "string"},
    {"name": "scrts_grntr_nm", "data_type": "string"},
    {"name": "isr_tin", "data_type": "string"},
    {"name": "rat_intrst_desc", "data_type": "string"},
    {"name": "cpn_nxt_dt", "data_type": "date"},
    {"name": "acg_acr_rph_intrst_ntrl_acc_id", "data_type": "string"},
    {"name": "acg_ifrs_exh_pos_ntrl_acc_id", "data_type": "string"},
    {"name": "acg_ifrs_off_bal_intrst_ntrl_acc_id", "data_type": "string"},
    {"name": "acg_ifrs_rvsd_intrst_ntrl_acc_id", "data_type": "string"},
    {"name": "acg_ifrs_rsv_imp_ntrl_acc_id", "data_type": "string"},
    {"name": "acg_ifrs_acd_fv_neg_ntrl_acc_id", "data_type": "string"},
    {"name": "acg_ifrs_acd_fv_pstv_ntrl_acc_id", "data_type": "string"},
    {"name": "acg_ifrs_fv_pstv_ntrl_acc_id", "data_type": "string"},
    {"name": "acg_ifrs_fv_neg_ntrl_acc_id", "data_type": "string"},
    {"name": "acg_ifrs_acr_rph_intrst_ntrl_acc_id", "data_type": "string"},
    {"name": "acg_ifrs_rph_ntrl_acc_id", "data_type": "string"},
    {"name": "qte_ngt_typ", "data_type": "string"},
    {"name": "assts_sec_typ", "data_type": "string"},
    {"name": "hqla_typ", "data_type": "string"},
    {"name": "ent_typ", "data_type": "string"},
    {"name": "ori_cntrct_id", "data_type": "string"},
    {"name": "acg_nca_fv_ntrl_acc_id", "data_type": "string"},
    {"name": "acg_npv_hdg_ntrl_acc_id", "data_type": "string"},
    {"name": "acg_ifrs_npv_hdg_ntrl_acc_id", "data_type": "string"},
    {"name": "acg_cap_acc_id", "data_type": "string"},
    {"name": "acg_intrst_acc_id", "data_type": "string"},
    {"name": "acg_acr_rvsd_intrst_acc_id", "data_type": "string"},
    {"name": "acg_acr_rph_intrst_acc_id", "data_type": "string"},
    {"name": "acg_rph_acc_id", "data_type": "string"},
    {"name": "acg_rsv_imp_acc_id", "data_type": "string"},
    {"name": "acg_acd_fv_neg_acc_id", "data_type": "string"},
    {"name": "acg_acd_fv_pstv_acc_id", "data_type": "string"},
    {"name": "acg_fv_pstv_acc_id", "data_type": "string"},
    {"name": "acg_fv_neg_acc_id", "data_type": "string"},
    {"name": "acg_imp_acc_id", "data_type": "string"},
    {"name": "acg_exh_pos_acc_id", "data_type": "string"},
    {"name": "acg_npv_hdg_acc_id", "data_type": "string"},
    {"name": "acg_off_bal_intrst_acc_id", "data_type": "string"},
    {"name": "rsdl_mat", "data_type": "int"},
    {"name": "fxd_frq_dys_nr", "data_type": "int"},
    {"name": "dur_per", "data_type": "decimal(38,17)"},
    {"name": "mdf_dur_per", "data_type": "decimal(38,17)"},
    {"name": "clbl_ptbl_ntc_dys_nr", "data_type": "int"},
    {"name": "reprc_dys_nr", "data_type": "int"},
    {"name": "scrts_cntrct_dys_nr", "data_type": "int"},
    {"name": "mnth_crdt_rsk_fv_chg_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "acd_crdt_rsk_fv_chg_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "ori_mkt_rat", "data_type": "decimal(38,17)"},
    {"name": "mkt_rat", "data_type": "decimal(38,17)"},
    {"name": "od_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "imp_cl_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "forei_exh_rsv_neg_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "forei_exh_rsv_pstv_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "scrts_col_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "cot_unt_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "qte_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "pgm_cd", "data_type": "string"},
    {"name": "cst_iss_lbl", "data_type": "decimal(38,17)"},
    {"name": "ivs_ent_ln", "data_type": "string"},
    {"name": "ivs_ent_ctry", "data_type": "string"},
    {"name": "ivs_ent_tin", "data_type": "string"},
    {"name": "sctr", "data_type": "string"},
    {"name": "ivs_ent_prty_id", "data_type": "string"},
    {"name": "lei_ivs_ent_doc_id", "data_type": "string"},
    {"name": "grntr_desc", "data_type": "string"},
    {"name": "ctry_res_grntr", "data_type": "string"},
    {"name": "grntr_tin", "data_type": "string"},
    {"name": "sctr_grntr", "data_type": "string"},
    {"name": "grntr_ini_dt", "data_type": "date"},
    {"name": "lei_grntr_doc_id", "data_type": "string"},
    {"name": "prty_nm", "data_type": "string"},
    {"name": "isr_sctr", "data_type": "string"},
    {"name": "lgd", "data_type": "decimal(38,17)"},
    {"name": "drwn_rsk_wght", "data_type": "decimal(38,17)"},
    {"name": "rsk_pos_cl_mtd", "data_type": "string"},
    {"name": "cla_corep_desc", "data_type": "string"},
    {"name": "prtf_rsk_desc", "data_type": "string"},
    {"name": "poci_idc_flg", "data_type": "int"},
    {"name": "ivs_ent_acg_std_cd", "data_type": "string"},
    {"name": "isr_ste", "data_type": "string"},
    {"name": "isr_ste_dt", "data_type": "date"},
    {"name": "rcvy_amnt", "data_type": "decimal(38,17)"},
    {"name": "fi_ivs_ent_cd", "data_type": "string"},
    {"name": "fi_grntr_cd", "data_type": "string"},
    {"name": "acg_ifrs_mnth_crdt_rsk_fv_chg_ntrl_acc_id", "data_type": "string"},
    {"name": "nipc_isr_doc_id", "data_type": "string"},
    {"name": "sicr_12m_flag", "data_type": "string"},
    {"name": "nte_eur_amnt", "data_type": "decimal(38,17)"},
    {"name": "isr_doc_id", "data_type": "string"},
    {"name": "isr_doc_typ", "data_type": "string"},
    {"name": "ivs_doc_typ", "data_type": "string"},
    {"name": "inv_isr_cd_typ", "data_type": "string"},
    {"name": "gar_cd_typ", "data_type": "string"},
    {"name": "deleted_flag", "data_type": "int"}
]

partition_cols = ["yr_mnth_nr"]

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

StatementMeta(, adcaaffa-a0e5-488d-87dc-e790547f59cc, 8, Finished, Available, Finished, False)

Table created - Name: dp_lgl_reg_fact_scrts; Type: TABLE; Partitions: ['yr_mnth_nr']


Checkpoint deleted - guid: 8e6048a5-dbf5-4732-ad1f-0266bb498945


##### **dp_lgl_reg_prty**

In [ ]:
guids = {"34ed999d-4b87-441f-be13-3260778980d2"}
source_table_name = "dp_lgl_reg_prty"

columns_list = [
    {"name": "yr_mnth_nr", "data_type": "int not null"},
    {"name": "ref_dt", "data_type": "date not null"},
    {"name": "prcss_dt", "data_type": "timestamp not null"},
    {"name": "prty_id", "data_type": "string not null"},
    {"name": "prty_ent_id", "data_type": "string not null"},
    {"name": "prty_inl_id", "data_type": "int not null"},
    {"name": "prty_nm", "data_type": "string"},

    {"name": "prty_tin", "data_type": "string"},
    {"name": "prty_tin_iss_start_dt", "data_type": "timestamp"},
    {"name": "prty_tin_iss_end_dt", "data_type": "timestamp"},

    {"name": "prty_lei_cd", "data_type": "string"},
    {"name": "prty_lei_iss_start_dt", "data_type": "timestamp"},
    {"name": "prty_lei_iss_end_dt", "data_type": "timestamp"},

    {"name": "prty_brth_dt", "data_type": "date"},
    {"name": "prty_gndr_cd", "data_type": "string"},
    {"name": "prty_gndr_ln", "data_type": "string"},
    {"name": "prty_prfsn_sit_cd", "data_type": "string"},
    {"name": "prty_prfsn_sit_ln", "data_type": "string"},
    {"name": "prty_hh_nr", "data_type": "int"},
    {"name": "prty_edu_lvl_cd", "data_type": "string"},
    {"name": "prty_edu_lvl_ln", "data_type": "string"},
    {"name": "prty_psn_typ", "data_type": "string"},

    {"name": "prty_lgl_frm_typ", "data_type": "int"},
    {"name": "prty_lgl_frm_ln", "data_type": "string"},

    {"name": "prty_cat_cd", "data_type": "int"},
    {"name": "prty_cat_ln", "data_type": "string"},

    {"name": "prty_fscl_ctry_cd", "data_type": "int"},
    {"name": "prty_fscl_ctry_ln", "data_type": "string"},
    {"name": "prty_fscl_ctry_iso2_ln", "data_type": "string"},
    {"name": "prty_fscl_ctry_iso3_ln", "data_type": "string"},

    {"name": "prty_fscl_adrs_strt_ln", "data_type": "string"},
    {"name": "prty_fscl_adrs_loc_ln", "data_type": "string"},
    {"name": "prty_fscl_adrs_pst_cd", "data_type": "string"},

    {"name": "prty_ntlt_ctry_cd", "data_type": "string"},
    {"name": "prty_ntlt_ctry_ln", "data_type": "string"},
    {"name": "prty_ntlt_ctry_iso2_ln", "data_type": "string"},
    {"name": "prty_ntlt_ctry_iso3_ln", "data_type": "string"},

    {"name": "prty_sctr_cd", "data_type": "string"},
    {"name": "prty_sctr_ln", "data_type": "string"},

    {"name": "prty_eco_actvty_cd", "data_type": "int"},
    {"name": "prty_eco_actvty_ln", "data_type": "string"},

    {"name": "prty_nace_cd", "data_type": "string"},
    {"name": "prty_nace_ln", "data_type": "string"},

    {"name": "sgm_rsk_cd", "data_type": "string"},
    {"name": "sgm_rsk_ln", "data_type": "string"},

    {"name": "hol_bkg_ent_id", "data_type": "string"},
    {"name": "intra_grp_flg", "data_type": "int"},

    {"name": "prty_sgm_cd", "data_type": "int"},
    {"name": "prty_sgm_ln", "data_type": "string"},
    {"name": "prty_scy_sgm_cd", "data_type": "int"},
    {"name": "prty_scy_sgm_ln", "data_type": "string"}
]

partition_cols = ["yr_mnth_nr"]

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

StatementMeta(, 31cc9732-219c-4a6a-8461-7120858f1259, 13, Finished, Available, Finished, False)

##### **dp_lgl_reg_dim_ent_hier**

In [ ]:
guids = {"a54cfbe9-ca6d-4e45-a7ed-a52e9cd07b35"}
source_table_name = "dp_lgl_reg_dim_ent_hier"
                 
columns_list = [
    {'name': 'ingestion_date', 'data_type': 'date not null'},
    {"name": "ref_dt", "data_type": "date not null"},
    {"name": "ent_id", "data_type": "string not null"},
    {"name": "ent_lvl_2_id", "data_type": "string"},
    {"name": "ent_lvl_3_id", "data_type": "string"},
    {"name": "ent_lvl_4_id", "data_type": "string"},
    {"name": "ent_lvl_5_id", "data_type": "string"},
    {"name": "ent_perim_cd", "data_type": "string"},
    {"name": "ent_perim_rpg_cd", "data_type": "string"},
    {"name": "ent_perim_rpg_bpce_cd", "data_type": "string"},
]

partition_cols = ["ref_dt"]

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

StatementMeta(, 31cc9732-219c-4a6a-8461-7120858f1259, 14, Finished, Available, Finished, False)

Table dp_lgl_reg_dim_ent_hier already exists. Skipping.


##### **dp_lgl_reg_int_ent_prm**

In [ ]:
guids = {"123ab7d2-d3bf-4030-9cd5-538fd41a1a55"}
source_table_name = "dp_lgl_reg_int_ent_prm"
                 
columns_list = [
    {'name': 'ingestion_date', 'data_type': 'date not null'},
    {"name": "ref_dt", "data_type": "date not null"},
    {"name": "rlt_ent_id", "data_type": "string not null"},
    {"name": "ent_id", "data_type": "string not null"},
    {"name": "ent_rlt_lvl_nr", "data_type": "int"},
    {"name": "ent_perim_cd", "data_type": "string"},
    {"name": "ent_elimin_flg", "data_type": "int"},
 
]

partition_cols = ["ref_dt"]

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

StatementMeta(, 31cc9732-219c-4a6a-8461-7120858f1259, 15, Finished, Available, Finished, False)

Table dp_lgl_reg_int_ent_prm already exists. Skipping.


##### **dp_lgl_reg_int_intra_grp**

In [ ]:
guids = {"3eea44aa-469a-492e-a498-2e0b48694048"}
source_table_name = "dp_lgl_reg_int_intra_grp"
                 
columns_list = [
     {'name': 'ingestion_date', 'data_type': 'date not null'},
     {"name": "ref_dt", "data_type": "date not null"},
     {"name": "src_id", "data_type": "string not null"},
     {"name": "instrmnt_id", "data_type": "string not null"},
     {"name": "prty_id", "data_type": "string not null"},
     {"name": "ent_id", "data_type": "string not null"},
     {"name": "bkg_ent_id", "data_type": "string"},
     {"name": "elimin_ent_id", "data_type": "string"},
     {"name": "inst_prty_rlt_typ", "data_type": "string"},
     {"name": "inst_prty_rlt_typ_ln", "data_type": "string"}

]

partition_cols = ["ref_dt"]

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

StatementMeta(, 31cc9732-219c-4a6a-8461-7120858f1259, 16, Finished, Available, Finished, False)

Table dp_lgl_reg_int_intra_grp already exists. Skipping.


##### **dp_lgl_reg_acg_inst_csl_ldgr**

In [ ]:
guids = {"128145e7-81e6-4770-85f0-1444689d8936"}
source_table_name = "dp_lgl_reg_acg_inst_csl_ldgr"
                 
columns_list = [
     {'name': 'prcss_dt', 'data_type': 'timestamp not null'},
     {"name": "bal_dt", "data_type": "date not null"},
     {"name": "ent_id", "data_type": "string not null"},
     {"name": "rlt_ent_id", "data_type": "string not null"},
     {"name": "instrmnt_id", "data_type": "string not null"},
     {"name": "acg_fll_acc_id", "data_type": "string not null"},
     {"name": "acg_cfm_subapl_ref_id", "data_type": "string"},
     {"name": "bnk_strctr_id", "data_type": "string"},
     {"name": "acg_bal_eur_amnt", "data_type": "DECIMAL(38,18)"},
     {"name": "csl_typ_cd", "data_type": "string"}
]

partition_cols = ["bal_dt"]

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

StatementMeta(, 31cc9732-219c-4a6a-8461-7120858f1259, 17, Finished, Available, Finished, False)

Table dp_lgl_reg_acg_inst_csl_ldgr already exists. Skipping.


##### **dp_lgl_reg_int_acg_csl_adj**

In [ ]:
guids = {"afd6094b-f54b-425f-9402-5e0fa4b71ac9"}
source_table_name = "dp_lgl_reg_int_acg_csl_adj"
                 
columns_list = [
     {'name': 'prcss_dt', 'data_type': 'timestamp not null'},
     {"name": "bal_dt", "data_type": "date not null"},
     {"name": "ent_id", "data_type": "string not null"},
     {"name": "acg_fll_acc_id", "data_type": "string not null"},
     {"name": "amnt_cntrct_ldgr", "data_type": "DECIMAL(38,18)"},
     {"name": "amnt_csl_ldgr", "data_type": "DECIMAL(38,18)"},
     {"name": "amnt_diff", "data_type": "DECIMAL(38,18)"},     
     {"name": "instrmnt_def_id", "data_type": "string"}
]

partition_cols = ["bal_dt"]

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

StatementMeta(, 31cc9732-219c-4a6a-8461-7120858f1259, 18, Finished, Available, Finished, False)

Table dp_lgl_reg_int_acg_csl_adj already exists. Skipping.


##### **dp_lgl_reg_acg_csl_rec**

In [ ]:
guids = {"b69226e4-e961-46d5-a7d4-9512b7520a79"}
source_table_name = "dp_lgl_reg_acg_csl_rec"
                 
columns_list = [
     {'name': 'prcss_dt', 'data_type': 'timestamp not null'},
     {"name": "bal_dt", "data_type": "date not null"},
     {"name": "ent_id", "data_type": "string not null"},
     {"name": "rlt_ent_id", "data_type": "string not null"},
     {"name": "instrmnt_id", "data_type": "string not null"},
     {"name": "acg_fll_acc_id", "data_type": "string not null"},
     {"name": "acg_cfm_subapl_ref_id", "data_type": "string"},
     {"name": "stc_id", "data_type": "string"},
     {"name": "acg_bal_eur_amnt", "data_type": "DECIMAL(38,18)"},
     {"name": "rec_typ_cd", "data_type": "string"}
    
]

partition_cols = ["bal_dt", "rlt_ent_id"]

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

StatementMeta(, 31cc9732-219c-4a6a-8461-7120858f1259, 19, Finished, Available, Finished, False)

Table dp_lgl_reg_acg_csl_rec already exists. Skipping.


##### **dp_lgl_reg_int_acg_msr**

In [5]:
guids = {"d7b0d1a5-e622-4a45-a75f-8b4adb4ec997"}
source_table_name = "dp_lgl_reg_int_acg_msr"
                 
columns_list = [
     {'name': 'prcss_dt', 'data_type': 'timestamp not null'},
     {"name": "ref_dt", "data_type": "date not null"},
     {"name": "ent_id", "data_type": "string not null"},
     {"name": "rlt_ent_id", "data_type": "string not null"},
     {"name": "instrmnt_id", "data_type": "string not null"},
     {"name": "acg_amnt_nm", "data_type": "string"},
     {"name": "acg_fll_acc_id", "data_type": "string not null"},
     {"name": "acg_bal_eur_amnt", "data_type": "DECIMAL(38,18)"}
    
]

partition_cols = ["ref_dt", "rlt_ent_id"]

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

StatementMeta(, e12b3b96-b328-4cf6-b68e-45277003611b, 8, Finished, Available, Finished, False)

Table created - Name: dp_lgl_reg_int_acg_msr; Type: TABLE; Partitions: ['ref_dt', 'rlt_ent_id']
Checkpoint deleted - guid: d7b0d1a5-e622-4a45-a75f-8b4adb4ec997


##### **dp_lgl_reg_dim_drvt_inst**

In [ ]:
guids = {"ff6649a0-096f-4e30-9104-4916146ebb48"}
source_table_name = "dp_lgl_reg_dim_drvt_inst"
                 
columns_list = [
    {"name": "prcss_dt", "data_type": "timestamp not null"},
    {"name": "ref_dt", "data_type": "date not null"},
    {"name": "instrmnt_id", "data_type": "string not null"},
    {"name": "prtf_nm", "data_type": "string"},
    {"name": "hier_lvl_nr", "data_type": "int"},
    {"name": "pdt_id", "data_type": "string"},
    {"name": "inst_typ", "data_type": "string"},
    {"name": "mkt_typ", "data_type": "string"},
    {"name": "rsk_typ", "data_type": "string"},
    {"name": "cap_mkt_bra_nm", "data_type": "string"},
    {"name": "cap_mkt_buy_flg", "data_type": "int"},
    {"name": "csp_dtf_floating_buy_nm", "data_type": "string"},
    {"name": "csp_dtf_floating_sell_nm", "data_type": "string"},
    {"name": "csp_dtf_base_buy", "data_type": "string"},
    {"name": "csp_dtf_base_sell", "data_type": "string"},
    {"name": "csp_dtf_base_quote_far_buy", "data_type": "string"},
    {"name": "csp_dtf_base_quote_far_sell", "data_type": "string"},
    {"name": "csp_dtf_base_quote_near_buy", "data_type": "string"},
    {"name": "csp_dtf_base_quote_near_sell", "data_type": "string"},
    {"name": "csp_dtf_base_quote_buy", "data_type": "string"},
    {"name": "csp_dtf_base_quote_sell", "data_type": "string"},
    {"name": "csp_dtf_ccy_far_buy", "data_type": "string"},
    {"name": "csp_dtf_ccy_far_sell", "data_type": "string"},
    {"name": "csp_dtf_ccy_near_buy", "data_type": "string"},
    {"name": "csp_dtf_ccy_near_sell", "data_type": "string"},
    {"name": "csp_dtf_ccy_buy", "data_type": "string"},
    {"name": "csp_dtf_ccy_sell", "data_type": "string"},
    {"name": "cntrct_ext_id", "data_type": "string"},
    {"name": "csp_dtf_csh_flw_amnt_cnb", "data_type": "decimal(38,18)"},
    {"name": "csp_dtf_csh_flw_amnt_cns", "data_type": "decimal(38,18)"},
    {"name": "csp_dtf_floating_rate_buy", "data_type": "decimal(38,18)"},
    {"name": "csp_dtf_floating_rate_sell", "data_type": "decimal(38,18)"},
    {"name": "csp_dtf_rate_buy", "data_type": "decimal(38,18)"},
    {"name": "csp_dtf_rate_sell", "data_type": "decimal(38,18)"},
    {"name": "csp_dtf_spread_fixed_rate_buy", "data_type": "decimal(38,18)"},
    {"name": "csp_dtf_spread_fixed_rate_sell", "data_type": "decimal(38,18)"},
    {"name": "csp_dtf_csh_flw_pym_end_dt_clb", "data_type": "timestamp"},
    {"name": "csp_dtf_csh_flw_pym_end_dt_cls", "data_type": "timestamp"},
    {"name": "csp_dtf_csh_flw_pym_end_dt_cnb", "data_type": "timestamp"},
    {"name": "csp_dtf_csh_flw_pym_end_dt_cns", "data_type": "timestamp"},
    {"name": "drv_lmt_id", "data_type": "string"},
    {"name": "drv_macr_hdg_ln", "data_type": "string"},
    {"name": "drv_mcr_hdg_ln", "data_type": "string"},
    {"name": "drv_sttlmnt_dt", "data_type": "string"},
    {"name": "ent_nm", "data_type": "string"},
    {"name": "fx_far_prim_eur_amnt", "data_type": "decimal(38,18)"},
    {"name": "fx_far_sec_eur_amnt", "data_type": "decimal(38,18)"},
    {"name": "notnl_eur_amnt", "data_type": "decimal(38,18)"},
    {"name": "pay_leg_princpl_eur_amnt", "data_type": "decimal(38,18)"},
    {"name": "rcv_leg_princpl_eur_amnt", "data_type": "decimal(38,18)"},
    {"name": "csp_dtf_amnt_eur_buy", "data_type": "decimal(38,18)"},
    {"name": "csp_dtf_amnt_eur_sell", "data_type": "decimal(38,18)"},
    {"name": "csp_dtf_amnt_ori_buy", "data_type": "decimal(38,18)"},
    {"name": "csp_dtf_amnt_ori_sell", "data_type": "decimal(38,18)"},
    {"name": "fx_spl_cur_cd", "data_type": "string"},
    {"name": "fx_spl_rt", "data_type": "decimal(38,18)"},
    {"name": "fxndf_rst_prc_amnt", "data_type": "decimal(38,18)"},
    {"name": "fxswp_far_allin_rt", "data_type": "decimal(38,18)"},
    {"name": "fxswp_near_allin_rt", "data_type": "decimal(38,18)"},
    {"name": "idrf", "data_type": "string"},
    {"name": "inst_mat_dt", "data_type": "string"},
    {"name": "csp_dtf_matrd", "data_type": "string"},
    {"name": "pay_leg_princpl_cur", "data_type": "string"},
    {"name": "csp_dtf_amnt_sell_far_leg_eur", "data_type": "decimal(38,18)"},
    {"name": "csp_dtf_amnt_buy_far_leg_eur", "data_type": "decimal(38,18)"},
    {"name": "csp_dtf_amnt_buy_near_leg_eur", "data_type": "decimal(38,18)"},
    {"name": "csp_dtf_amnt_sell_near_leg_eur", "data_type": "decimal(38,18)"},
    {"name": "csp_dtf_amnt_sell_far_leg_ori", "data_type": "decimal(38,18)"},
    {"name": "csp_dtf_amnt_buy_far_leg_ori", "data_type": "decimal(38,18)"},
    {"name": "csp_dtf_amnt_buy_near_leg_ori", "data_type": "decimal(38,18)"},
    {"name": "csp_dtf_amnt_sell_near_leg_ori", "data_type": "decimal(38,18)"},
    {"name": "prty_id", "data_type": "string"},
    {"name": "spc_drv_col_typ", "data_type": "string"},
    {"name": "spc_drv_trmt_ln", "data_type": "string"},
    {"name": "trd_boo_ln", "data_type": "string"},
    {"name": "trd_ctp_ext_id", "data_type": "string"},
    {"name": "trd_dt", "data_type": "string"},
    {"name": "trd_ext_ref_id", "data_type": "string"},
    {"name": "trd_id", "data_type": "string"},
    {"name": "trd_mir_boo_ln", "data_type": "string"},
    {"name": "trd_mir_id", "data_type": "string"},
    {"name": "trd_pdt_typ", "data_type": "string"},
    {"name": "csp_dtf_csh_flw_fnl_rt_cnb", "data_type": "decimal(38,18)"},
    {"name": "csp_dtf_csh_flw_fnl_rt_cns", "data_type": "decimal(38,18)"},
    {"name": "repo_isn_cd", "data_type": "string"},
    {"name": "trd_prc_rt", "data_type": "decimal(38,18)"},
    {"name": "trd_stts", "data_type": "string"},
    {"name": "quantity_ori_amnt", "data_type": "decimal(38,18)"},
    {"name": "csp_dtf_days_sttlmnt", "data_type": "bigint"},
    {"name": "csp_dtf_deal_perd", "data_type": "bigint"},
    {"name": "csp_dtf_refrnc_buy_ccy", "data_type": "decimal(38,18)"},
    {"name": "csp_dtf_refrnc_sell_ccy", "data_type": "decimal(38,18)"},
    {"name": "csp_dtf_resdl_mat_perd", "data_type": "bigint"},
    {"name": "csp_dtf_dervd_embtd", "data_type": "string"},
    {"name": "csp_dtf_mat_perd", "data_type": "bigint"},
    {"name": "flg_vivo", "data_type": "int"},
    {"name": "csp_dtf_flg_endofmonth", "data_type": "date"},
]

partition_cols = ["ref_dt"]

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

StatementMeta(, 31cc9732-219c-4a6a-8461-7120858f1259, 20, Finished, Available, Finished, False)

Table dp_lgl_reg_dim_drvt_inst already exists. Skipping.


##### **dp_lgl_reg_acg_csl_drvt**

In [ ]:
# guids = {"fde2326c-de44-4239-8be3-0aaee6a53346"}
# source_table_name = "dp_lgl_reg_acg_csl_drvt"
                 
# columns_list = [
#     {"name": "prcss_dt", "data_type": "timestamp not null"},
#     {"name": "ref_dt", "data_type": "date not null"},
#     {"name": "instrmnt_id", "data_type": "string not null"},
#     {"name": "ent_id", "data_type": "string"},
#     {"name": "rlt_ent_id", "data_type": "string"},
#     {"name": "prtf_nm", "data_type": "string"},
#     {"name": "hier_lvl_nr", "data_type": "int"},
#     {"name": "pdt_id", "data_type": "string"},
#     {"name": "inst_typ", "data_type": "string"},
#     {"name": "mkt_typ", "data_type": "string"},
#     {"name": "rsk_typ", "data_type": "string"},
#     {"name": "csp_dtf_last_cupn_buy", "data_type": "date"},
#     {"name": "csp_dtf_last_cupn_sell", "data_type": "date"},
#     {"name": "csp_dtf_next_cupn_buy", "data_type": "date"},
#     {"name": "csp_dtf_next_cupn_sell", "data_type": "date"},
#     {"name": "csh_flw_typ", "data_type": "string"},
#     {"name": "csh_flw_intrst_sprd_rt", "data_type": "decimal(38,18)"},
#     {"name": "csh_flw_amnt", "data_type": "decimal(38,18)"},
#     {"name": "csh_flw_rt", "data_type": "decimal(38,18)"},
#     {"name": "csh_flw_rat_tnr_typ", "data_type": "string"},
#     {"name": "csh_flw_cls_ln", "data_type": "string"},
#     {"name": "csp_dtf_mat_dt", "data_type": "date"},
#     {"name": "drv_sttlmnt_dt", "data_type": "date"},
#     {"name": "drv_lmt_id", "data_type": "string"},
#     {"name": "csh_flw_fnl_rt", "data_type": "decimal(38,18)"},
#     {"name": "csp_dtf_days_settlement", "data_type": "int"},
#     {"name": "fx_spl_rt", "data_type": "decimal(38,18)"},
#     {"name": "fx_spl_cur_cd", "data_type": "string"},
#     {"name": "fx_spl_sc_cur_cd", "data_type": "string"},
#     {"name": "fxswp_far_allin_rt", "data_type": "decimal(38,18)"},
#     {"name": "fxswp_near_allin_rt", "data_type": "decimal(38,18)"},
#     {"name": "fxswp_near_dt", "data_type": "date"},
#     {"name": "cap_mkt_buy_flg", "data_type": "string"},
#     {"name": "cap_mkt_bra_nm", "data_type": "string"},
#     {"name": "cap_mkt_uti_cd", "data_type": "string"},
#     {"name": "cap_mkt_prc_trd_day_ln", "data_type": "string"},
#     {"name": "cap_mkt_prc_def_idx_ln", "data_type": "string"},
#     {"name": "csp_dtf_dervd_embtd", "data_type": "string"},
#     {"name": "trd_id", "data_type": "string"},
#     {"name": "trd_ext_ref_id", "data_type": "string"},
#     {"name": "trd_boo_ln", "data_type": "string"},
#     {"name": "trd_pur_shrt_nm", "data_type": "string"},
#     {"name": "trd_dt", "data_type": "date"},
#     {"name": "trd_prc_rt", "data_type": "decimal(38,18)"},
#     {"name": "trd_pdt_typ", "data_type": "string"},
#     {"name": "trd_ctp_ext_id", "data_type": "string"},
#     {"name": "trd_stts", "data_type": "string"},
#     {"name": "trd_mir_id", "data_type": "string"},
#     {"name": "trd_mir_boo_ln", "data_type": "string"},
#     {"name": "trd_start_dt", "data_type": "date"},
#     {"name": "irs_cirs_leg_sttlmnt_cur_cd", "data_type": "string"},
#     {"name": "irs_cirs_leg_pym_cl_typ", "data_type": "string"},
#     {"name": "irs_cirs_leg_start_dt", "data_type": "date"},
#     {"name": "irs_cirs_leg_flt_ln", "data_type": "string"},
#     {"name": "irs_cirs_leg_intrst_frq_nm", "data_type": "string"},
#     {"name": "repo_isn_cd", "data_type": "string"},
#     {"name": "csp_dtf_mcr_macr_hdg", "data_type": "string"},
#     {"name": "spc_drv_trmt_ln", "data_type": "date"},
#     {"name": "spc_drv_col_typ", "data_type": "string"},
#     {"name": "drv_mcr_hdg_ln", "data_type": "string"},
#     {"name": "drv_macr_hdg_ln", "data_type": "string"},
#     {"name": "cntrct_ext_id", "data_type": "string"},
#     {"name": "prnt_cntrct_id", "data_type": "string"},
#     {"name": "inst_pdt_id", "data_type": "string"},
#     {"name": "inst_cur_cd", "data_type": "string"},
#     {"name": "csp_dtf_residual_days_mat", "data_type": "int"},
#     {"name": "princpl_amnt_cur", "data_type": "string"},
#     {"name": "princpl_eur_amnt", "data_type": "decimal(38,18)"},
#     {"name": "princpl_ori_amnt", "data_type": "decimal(38,18)"},
#     {"name": "prim_cur", "data_type": "string"},
#     {"name": "prim_eur_amnt", "data_type": "decimal(38,18)"},
#     {"name": "prim_ori_amnt", "data_type": "decimal(38,18)"},
#     {"name": "sec_cur", "data_type": "string"},
#     {"name": "sec_eur_amnt", "data_type": "decimal(38,18)"},
#     {"name": "sec_ori_amnt", "data_type": "decimal(38,18)"},
#     {"name": "notnl_cur", "data_type": "string"},
#     {"name": "notnl_eur_amnt", "data_type": "decimal(38,18)"},
#     {"name": "notnl_ori_amnt", "data_type": "decimal(38,18)"},
#     {"name": "fx_far_prim_cur", "data_type": "string"},
#     {"name": "fx_far_prim_eur_amnt", "data_type": "decimal(38,18)"},
#     {"name": "fx_far_prim_ori_amnt", "data_type": "decimal(38,18)"},
#     {"name": "fx_far_sec_cur", "data_type": "string"},
#     {"name": "fx_far_sec_eur_amnt", "data_type": "decimal(38,18)"},
#     {"name": "fx_far_sec_ori_amnt", "data_type": "decimal(38,18)"},
#     {"name": "settlmnt_ori_amnt", "data_type": "decimal(38,18)"},
#     {"name": "settlmnt_eur_amnt", "data_type": "decimal(38,18)"},
#     {"name": "settlmnt_cur", "data_type": "string"},
#     {"name": "rcv_leg_princpl_cur", "data_type": "string"},
#     {"name": "rcv_leg_princpl_eur_amnt", "data_type": "decimal(38,18)"},
#     {"name": "rcv_leg_princpl_ori_amnt", "data_type": "decimal(38,18)"},
#     {"name": "pay_leg_princpl_cur", "data_type": "string"},
#     {"name": "pay_leg_princpl_eur_amnt", "data_type": "decimal(38,18)"},
#     {"name": "pay_leg_princpl_ori_amnt", "data_type": "decimal(38,18)"},
#     {"name": "csp_dtf_limited_group", "data_type": "string"},
#     {"name": "csp_dtf_underlying", "data_type": "string"},
#     {"name": "csp_dtf_period_deal", "data_type": "int"},
#     {"name": "csp_dtf_ccy_bought", "data_type": "string"},
#     {"name": "csp_dtf_ccy_sold", "data_type": "string"},
#     {"name": "csp_dtf_mat_period", "data_type": "int"},
#     {"name": "csp_dtf_deal_period", "data_type": "int"},
#     {"name": "prty_id", "data_type": "string"},
#     {"name": "prty_nm", "data_type": "string"},
#     {"name": "prty_ctry_cd", "data_type": "string"},
#     {"name": "bkg_ent_id", "data_type": "string"},
#     {"name": "prty_cat_ln", "data_type": "string"},
#     {"name": "prty_typ", "data_type": "string"},
#     {"name": "prty_rsk_typ", "data_type": "string"},
#     {"name": "acg_mkt_pay_acc_id", "data_type": "string"},
#     {"name": "acg_mkt_rec_acc_id", "data_type": "string"},
#     {"name": "acg_noc_pay_acc_id", "data_type": "string"},
#     {"name": "acg_noc_rec_acc_id", "data_type": "string"},
#     {"name": "acg_prem_acc_id", "data_type": "string"},
#     {"name": "acg_dis_acc_id", "data_type": "string"},
#     {"name": "acg_acr_intrst_pay_acc_id", "data_type": "string"},
#     {"name": "acg_acr_intrst_rec_acc_id", "data_type": "string"},
#     {"name": "acg_acr_intrst_neg_pay_acc_id", "data_type": "string"},
#     {"name": "acg_acr_intrst_neg_rec_acc_id", "data_type": "string"},
#     {"name": "acg_sttlmnt_nr_lg_pay_acc_id", "data_type": "string"},
#     {"name": "acg_sttlmnt_nr_lg_rec_acc_id", "data_type": "string"},
#     {"name": "acg_nr_pos_acc_id", "data_type": "string"},
#     {"name": "acg_fwd_pos_acc_id", "data_type": "string"},
#     {"name": "acg_sttlmnt_fwd_pay_acc_id", "data_type": "string"},
#     {"name": "acg_sttlmnt_fwd_rec_acc_id", "data_type": "string"},
#     {"name": "acg_cva_acc_id", "data_type": "string"},
#     {"name": "acg_mkt_pay_amnt", "data_type": "decimal(38,18)"},
#     {"name": "acg_mkt_rec_amnt", "data_type": "decimal(38,18)"},
#     {"name": "acg_noc_pay_amnt", "data_type": "decimal(38,18)"},
#     {"name": "acg_noc_rec_amnt", "data_type": "decimal(38,18)"},
#     {"name": "acg_prem_amnt", "data_type": "decimal(38,18)"},
#     {"name": "acg_dis_amnt", "data_type": "decimal(38,18)"},
#     {"name": "acg_acr_intrst_pay_amnt", "data_type": "decimal(38,18)"},
#     {"name": "acg_acr_intrst_rec_amnt", "data_type": "decimal(38,18)"},
#     {"name": "acg_acr_intrst_neg_pay_amnt", "data_type": "decimal(38,18)"},
#     {"name": "acg_acr_intrst_neg_rec_amnt", "data_type": "decimal(38,18)"},
#     {"name": "acg_sttlmnt_nr_lg_pay_amnt", "data_type": "decimal(38,18)"},
#     {"name": "acg_sttlmnt_nr_lg_rec_amnt", "data_type": "decimal(38,18)"},
#     {"name": "acg_nr_pos_amnt", "data_type": "decimal(38,18)"},
#     {"name": "acg_fwd_pos_amnt", "data_type": "decimal(38,18)"},
#     {"name": "acg_sttlmnt_fwd_pay_amnt", "data_type": "decimal(38,18)"},
#     {"name": "acg_sttlmnt_fwd_rec_amnt", "data_type": "decimal(38,18)"},
#     {"name": "acg_cva_amnt", "data_type": "decimal(38,18)"}
# ]

# partition_cols = ["ref_dt"]

# create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

StatementMeta(, 31cc9732-219c-4a6a-8461-7120858f1259, 21, Finished, Available, Finished, False)

##### **dp_lgl_reg_instrmnt_ori**

In [ ]:
guids = ["1b8c0841-6590-4420-a1b3-ab87d5d6673a"]
source_table_name = "dp_lgl_reg_instrmnt_ori"

columns_list = [
    # Control / Framework
    {"name": "del_flg", "data_type": "INT"},
    {"name": "rej_flg", "data_type": "INT"},

    # Claves (Join Keys)
    {"name": "ent_id", "data_type": "STRING NOT NULL"},
    {"name": "instrmnt_id", "data_type": "STRING NOT NULL"},
    
    # IDs y Códigos
    {"name": "cntrct_id", "data_type": "STRING"},
    {"name": "prnt_cntrct_id", "data_type": "STRING"},
    {"name": "cntrct_ext_id", "data_type": "STRING"},
    {"name": "apl_src_id", "data_type": "STRING"},
    {"name": "cntrct_cur_cd", "data_type": "STRING"},
    {"name": "inst_dtl_id", "data_type": "STRING"},
    {"name": "src_id", "data_type": "STRING"},
    {"name": "inst_pdt_id", "data_type": "STRING"},
    {"name": "inst_cur_cd", "data_type": "STRING"},
    {"name": "inst_ctry_cd", "data_type": "STRING"},
    {"name": "inst_cml_chnnl_cd", "data_type": "STRING"},
    {"name": "inst_assts_pur_cd", "data_type": "STRING"},
    {"name": "inst_assts_crdt_cat_cd", "data_type": "STRING"},
    {"name": "inst_assts_prot_cd", "data_type": "STRING"},
    {"name": "inst_assts_req_ins_cd", "data_type": "STRING"},
    
    # Fechas (Timestamps)
    {"name": "ins_act_flg", "data_type": "TIMESTAMP"},
    {"name": "inst_start_dt", "data_type": "TIMESTAMP"},
    {"name": "inst_end_dt", "data_type": "TIMESTAMP"},
    {"name": "inst_mat_dt", "data_type": "TIMESTAMP"},
    {"name": "inst_grc_per_start_dt", "data_type": "TIMESTAMP"},
    {"name": "inst_grc_per_end_dt", "data_type": "TIMESTAMP"},
    
    # Tipos numéricos e importes
    {"name": "inst_intrst_sprd_rt", "data_type": "DECIMAL(38,18)"},
    {"name": "inst_credtr_intrst_rt", "data_type": "DECIMAL(38,18)"},
    {"name": "inst_intrst_flr_rt", "data_type": "DECIMAL(38,18)"},
    {"name": "inst_ini_ori_amnt", "data_type": "DECIMAL(38,18)"},
    {"name": "inst_ini_eur_amnt", "data_type": "DECIMAL(38,18)"}
]

partition_cols = []

create_table(source_table_name, 'SCD22', columns_list, reset_list, partition_cols, guids)

StatementMeta(, 31cc9732-219c-4a6a-8461-7120858f1259, 22, Finished, Available, Finished, False)

Table created - Name: dp_lgl_reg_instrmnt_ori; Type: SCD22; Partitions: []
Checkpoint deleted - guid: 1b8c0841-6590-4420-a1b3-ab87d5d6673a


##### **dp_lgl_reg_prcss_rules**

In [ ]:
guids = {"a026b22c-c4eb-4291-96c9-70b070fc1640"}
source_table_name = 'dp_lgl_reg_prcss_rules'
columns_list = [
     {'name': 'prcss_nm',        'data_type': 'string NOT NULL'},
     {'name': 'del_flg',         'data_type': 'int NOT NULL'},
     {'name': 'rej_flg',         'data_type': 'int NOT NULL'},
     {'name': 'src_tbl',         'data_type': 'string NOT NULL'},
     {'name': 'att_nm',          'data_type': 'string'},
     {'name': 'prcss_rule_typ',  'data_type': 'string NOT NULL'},
     {'name': 'cde',             'data_type': 'string NOT NULL'},
     {'name': 'val',             'data_type': 'string'},
     {'name': 'usr_id',          'data_type': 'string'}
 ]

partition_cols=["prcss_nm"]

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

StatementMeta(, 31cc9732-219c-4a6a-8461-7120858f1259, 23, Finished, Available, Finished, False)

Table dp_lgl_reg_prcss_rules already exists. Skipping.


##### **dp_lgl_reg_cdelst**

In [ ]:
guids = {"33e89d42-bc6f-46e4-9a7e-ab0e491c5031"}
source_table_name = 'dp_lgl_reg_cdelst'
columns_list = [
    {'name': 'dmn_nm',  'data_type': 'STRING NOT NULL'},
    {'name': 'att_nm',  'data_type': 'STRING NOT NULL'},
    {'name': 'del_flg', 'data_type': 'INT NOT NULL'},
    {'name': 'rej_flg', 'data_type': 'INT NOT NULL'},
    {'name': 'dmn_typ', 'data_type': 'STRING NOT NULL'},
    {'name': 'cde',     'data_type': 'STRING NOT NULL'},
    {'name': 'val',     'data_type': 'STRING'},
    {'name': 'prnt',    'data_type': 'STRING'},
    {'name': 'lvl',     'data_type': 'INT NOT NULL'},
    {'name': 'src_typ', 'data_type': 'STRING NOT NULL'}
    ]

partition_cols=["dmn_nm", "dmn_typ"]

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

StatementMeta(, 31cc9732-219c-4a6a-8461-7120858f1259, 24, Finished, Available, Finished, False)

Table dp_lgl_reg_cdelst already exists. Skipping.


##### **dp_lgl_reg_sec_hld**

In [5]:
guids = {"d44d1959-dcad-4385-9e76-a024c19d80b2"}   
source_table_name = "dp_lgl_reg_sec_hld"

columns_list = [
    {"name": "report_date",            "data_type": "date not null"},
    # primary key
    {"name": "ent_id",                 "data_type": "string not null"},
    {"name": "isin_id",                "data_type": "string not null"},
    {"name": "instrmnt_id",            "data_type": "string not null"},
    # enr_sec
    {"name": "sec_id",                 "data_type": "string"},
    {"name": "sec_typ",                "data_type": "string"},
    {"name": "sec_iss_dt",             "data_type": "date"},
    {"name": "sec_mat_dt",             "data_type": "date"},
    {"name": "sec_fc_val_amnt",        "data_type": "DECIMAL(38,10)"},
    # enr_instrmnt_sec_hld
    {"name": "hld_src_id",             "data_type": "string"},
    {"name": "sec_hld_inst_typ",       "data_type": "string"},
    {"name": "sec_hld_nm",             "data_type": "string"},
    {"name": "sec_hld_prtf_typ",       "data_type": "string"},
    {"name": "hld_ref_start_dt",       "data_type": "timestamp"},
    # enr_instrmnt
    {"name": "instr_src_id",           "data_type": "string"},
    {"name": "inst_cur_cd",            "data_type": "string"},
    {"name": "inst_end_dt",            "data_type": "timestamp"},
    # enr_instrmnt_fin_pos
    {"name": "inst_fin_pos_eur_amnt",  "data_type": "DECIMAL(38,10)"},
    {"name": "pos_ref_start_dt",       "data_type": "date"},
    {"name": "pos_src_id",             "data_type": "string"},
    {"name": "pos_del_flg",            "data_type": "int"},
    {"name": "pos_fin_pos_typ",        "data_type": "string"}
]

partition_cols = ["ent_id", "isin_id", "instrmnt_id"]

create_table(source_table_name, 'SCD22', columns_list, reset_list, partition_cols, guids)

StatementMeta(, 7046c2f2-d03c-415d-aaf3-60a26991c97b, 9, Finished, Available, Finished, False)

Table created - Name: dp_lgl_reg_sec_hld; Type: SCD22; Partitions: ['ent_id', 'isin_id', 'instrmnt_id']


##### **dp_lgl_reg_sec_hld_cur**

In [7]:
guids = {"b3643329-fac9-4e13-b1c4-1c2f817fc335"} 
source_table_name = "dp_lgl_reg_sec_hld_cur"

columns_list = [
    {"name": "report_date",            "data_type": "date not null"},
    # primary key
    {"name": "ent_id",                 "data_type": "string not null"},
    {"name": "isin_id",                "data_type": "string not null"},
    {"name": "instrmnt_id",            "data_type": "string not null"},
    # satellite columns
    {"name": "sec_id",                 "data_type": "string"},
    {"name": "sec_typ",                "data_type": "string"},
    {"name": "sec_iss_dt",             "data_type": "date"},
    {"name": "sec_mat_dt",             "data_type": "date"},
    {"name": "sec_fc_val_amnt",        "data_type": "DECIMAL(38,10)"},
    {"name": "hld_src_id",             "data_type": "string"},
    {"name": "sec_hld_inst_typ",       "data_type": "string"},
    {"name": "sec_hld_nm",             "data_type": "string"},
    {"name": "sec_hld_prtf_typ",       "data_type": "string"},
    {"name": "hld_ref_start_dt",       "data_type": "timestamp"},
    {"name": "instr_src_id",           "data_type": "string"},
    {"name": "inst_cur_cd",            "data_type": "string"},
    {"name": "inst_end_dt",            "data_type": "timestamp"},
    {"name": "inst_fin_pos_eur_amnt",  "data_type": "DECIMAL(38,10)"},
    {"name": "pos_ref_start_dt",       "data_type": "date"},
    {"name": "pos_src_id",             "data_type": "string"},
    {"name": "pos_del_flg",            "data_type": "int"},
    {"name": "pos_fin_pos_typ",        "data_type": "string"},
    # dp_lgl_reg_rlt_instrmnt_prty
    {"name": "prty_id",                "data_type": "string"},
    {"name": "inst_prty_rlt_typ",      "data_type": "string"},
    {"name": "rlt_src_id",             "data_type": "string"},
    # enr_sec_inl_fin_acc
    {"name": "acg_fll_acc_id",         "data_type": "string"},
    {"name": "fa_fin_pos_typ",         "data_type": "int"},
    # enr_exh
    {"name": "issuance_exchange_rate", "data_type": "DECIMAL(38,10)"},
    {"name": "fx_ref_start_dt",        "data_type": "date"},
    {"name": "fx_rate_type",           "data_type": "string"}
]

partition_cols = ["ent_id", "isin_id", "instrmnt_id"]

create_table(source_table_name, 'SCD22', columns_list, reset_list, partition_cols, guids)



StatementMeta(, 7046c2f2-d03c-415d-aaf3-60a26991c97b, 11, Finished, Available, Finished, False)

Table created - Name: dp_lgl_reg_sec_hld_cur; Type: SCD22; Partitions: ['ent_id', 'isin_id', 'instrmnt_id']


#### **Relation Tables**
___

##### **dp_lgl_reg_rlt_instrmnt_prty**

In [ ]:
guids = {"0d574b2a-2508-4c17-be5b-21195a0007dc"}
source_table_name = "dp_lgl_reg_rlt_instrmnt_prty"

columns_list = [
    {"name": "yr_mnth_nr", "data_type": "int"},
    {"name": "ref_dt", "data_type": "date not null"},
    {"name": "ent_id", "data_type": "string not null"},
    {"name": "instrmnt_id", "data_type": "string not null"},
    {"name": "prty_id", "data_type": "string not null"},
    {"name": "src_id", "data_type": "string not null"},
    {"name": "inst_prty_rlt_typ", "data_type": "string not null"},
    {"name": "inst_prty_rlt_typ_ln", "data_type": "string"},
    {"name": "inst_prty_rlt_typ_nr", "data_type": "int"},
    {"name": "inst_prty_rlt_start_dt", "data_type": "timestamp"},
    {"name": "inst_prty_rlt_end_dt", "data_type": "timestamp"}
]

partition_cols = ["yr_mnth_nr"]

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

StatementMeta(, 31cc9732-219c-4a6a-8461-7120858f1259, 26, Finished, Available, Finished, False)

#### **DPM Engine Tables**
___
Tables written by the DPM engine notebooks (DE_ENR_NB_DPM_*). They are not loaded by a WorkerFactory process, so they have no checkpoint (empty `guids`).

##### **dp_lgl_reg_dpm_param_datapoint**

In [ ]:
guids = set()
source_table_name = "dp_lgl_reg_dpm_param_datapoint"

columns_list = [
    {"name": "framework_cd",                       "data_type": "string"},
    {"name": "datapoint_id",                       "data_type": "string"},
    {"name": "var_cd",                             "data_type": "string"},
    {"name": "datapoint_desc",                     "data_type": "string"},
    {"name": "datapoint_cl_typ",                   "data_type": "string"},
    {"name": "src_tbl_id",                         "data_type": "string"},
    {"name": "metric_fmla_ln",                     "data_type": "string"},
    {"name": "having_ln",                          "data_type": "string"},
    {"name": "per_expr_cd",                        "data_type": "string"},
    {"name": "excel_check_ln",                     "data_type": "string"},
    {"name": "src_file_nm",                        "data_type": "string"},
    {"name": "src_she_nm",                         "data_type": "string"},
    {"name": "src_row_nr",                         "data_type": "bigint"},
    {"name": "metadata_load_id",                   "data_type": "string"},
    {"name": "load_dt",                            "data_type": "timestamp"},
    {"name": "row_present_flg",                    "data_type": "int"},
    {"name": "dpm_vrs_dt",                         "data_type": "timestamp"},
    {"name": "dpm_ingestion_dt",                   "data_type": "timestamp"},
    {"name": "sk_dp_lgl_reg_dpm_param_datapoint",  "data_type": "bigint"},
    {"name": "ref_start_dt",                       "data_type": "timestamp"},
    {"name": "ref_end_dt",                         "data_type": "timestamp"},
    {"name": "ref_last_dt",                        "data_type": "timestamp"},
    {"name": "ctrl_start_dt",                      "data_type": "timestamp"},
    {"name": "ctrl_end_dt",                        "data_type": "timestamp"},
    {"name": "ctrl_updt_dt",                       "data_type": "timestamp"},
    {"name": "ctrl_state",                         "data_type": "int"}
]

partition_cols = []

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

##### **dp_lgl_reg_dpm_param_fltr**

In [ ]:
guids = set()
source_table_name = "dp_lgl_reg_dpm_param_fltr"

columns_list = [
    {"name": "datapoint_id",                  "data_type": "string"},
    {"name": "var_cd",                        "data_type": "string"},
    {"name": "src_column_nm",                 "data_type": "string"},
    {"name": "optr_cd",                       "data_type": "string"},
    {"name": "fltr_val_ln",                   "data_type": "string"},
    {"name": "excel_check_ln",                "data_type": "string"},
    {"name": "src_file_nm",                   "data_type": "string"},
    {"name": "src_she_nm",                    "data_type": "string"},
    {"name": "src_row_nr",                    "data_type": "bigint"},
    {"name": "metadata_load_id",              "data_type": "string"},
    {"name": "load_dt",                       "data_type": "timestamp"},
    {"name": "row_present_flg",               "data_type": "int"},
    {"name": "dpm_vrs_dt",                    "data_type": "timestamp"},
    {"name": "dpm_ingestion_dt",              "data_type": "timestamp"},
    {"name": "sk_dp_lgl_reg_dpm_param_fltr",  "data_type": "bigint"},
    {"name": "ref_start_dt",                  "data_type": "timestamp"},
    {"name": "ref_end_dt",                    "data_type": "timestamp"},
    {"name": "ref_last_dt",                   "data_type": "timestamp"},
    {"name": "ctrl_start_dt",                 "data_type": "timestamp"},
    {"name": "ctrl_end_dt",                   "data_type": "timestamp"},
    {"name": "ctrl_updt_dt",                  "data_type": "timestamp"},
    {"name": "ctrl_state",                    "data_type": "int"}
]

partition_cols = []

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

##### **dp_lgl_reg_dpm_param_ctlg**

In [ ]:
guids = set()
source_table_name = "dp_lgl_reg_dpm_param_ctlg"

columns_list = [
    {"name": "datapoint_id",                  "data_type": "string"},
    {"name": "framework_cd",                  "data_type": "string"},
    {"name": "anx_cd",                        "data_type": "string"},
    {"name": "tmplt_cd",                      "data_type": "string"},
    {"name": "row_cd",                        "data_type": "string"},
    {"name": "column_cd",                     "data_type": "string"},
    {"name": "row_desc",                      "data_type": "string"},
    {"name": "column_lvl_1_desc",             "data_type": "string"},
    {"name": "column_lvl_2_desc",             "data_type": "string"},
    {"name": "val_typ",                       "data_type": "string"},
    {"name": "excel_check_ln",                "data_type": "string"},
    {"name": "src_file_nm",                   "data_type": "string"},
    {"name": "src_she_nm",                    "data_type": "string"},
    {"name": "src_row_nr",                    "data_type": "bigint"},
    {"name": "metadata_load_id",              "data_type": "string"},
    {"name": "load_dt",                       "data_type": "timestamp"},
    {"name": "row_present_flg",               "data_type": "int"},
    {"name": "dpm_vrs_dt",                    "data_type": "timestamp"},
    {"name": "dpm_ingestion_dt",              "data_type": "timestamp"},
    {"name": "sk_dp_lgl_reg_dpm_param_ctlg",  "data_type": "bigint"},
    {"name": "ref_start_dt",                  "data_type": "timestamp"},
    {"name": "ref_end_dt",                    "data_type": "timestamp"},
    {"name": "ref_last_dt",                   "data_type": "timestamp"},
    {"name": "ctrl_start_dt",                 "data_type": "timestamp"},
    {"name": "ctrl_end_dt",                   "data_type": "timestamp"},
    {"name": "ctrl_updt_dt",                  "data_type": "timestamp"},
    {"name": "ctrl_state",                    "data_type": "int"}
]

partition_cols = []

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

##### **dp_lgl_reg_dpm_param_src_tbl**

In [ ]:
guids = set()
source_table_name = "dp_lgl_reg_dpm_param_src_tbl"

columns_list = [
    {"name": "src_tbl_id",                       "data_type": "string"},
    {"name": "phy_tbl_nm",                       "data_type": "string"},
    {"name": "per_column_nm",                    "data_type": "string"},
    {"name": "per_fmt_cd",                       "data_type": "string"},
    {"name": "ent_column_nm",                    "data_type": "string"},
    {"name": "ident_column_nm",                  "data_type": "string"},
    {"name": "subent_column_nm",                 "data_type": "string"},
    {"name": "ctry_column_nm",                   "data_type": "string"},
    {"name": "grain_id",                         "data_type": "string"},
    {"name": "src_file_nm",                      "data_type": "string"},
    {"name": "src_she_nm",                       "data_type": "string"},
    {"name": "src_row_nr",                       "data_type": "bigint"},
    {"name": "metadata_load_id",                 "data_type": "string"},
    {"name": "load_dt",                          "data_type": "timestamp"},
    {"name": "row_present_flg",                  "data_type": "int"},
    {"name": "dpm_vrs_dt",                       "data_type": "timestamp"},
    {"name": "dpm_ingestion_dt",                 "data_type": "timestamp"},
    {"name": "sk_dp_lgl_reg_dpm_param_src_tbl",  "data_type": "bigint"},
    {"name": "ref_start_dt",                     "data_type": "timestamp"},
    {"name": "ref_end_dt",                       "data_type": "timestamp"},
    {"name": "ref_last_dt",                      "data_type": "timestamp"},
    {"name": "ctrl_start_dt",                    "data_type": "timestamp"},
    {"name": "ctrl_end_dt",                      "data_type": "timestamp"},
    {"name": "ctrl_updt_dt",                     "data_type": "timestamp"},
    {"name": "ctrl_state",                       "data_type": "int"}
]

partition_cols = []

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

##### **dp_lgl_reg_dpm_param_iss_ctlg**

In [ ]:
guids = set()
source_table_name = "dp_lgl_reg_dpm_param_iss_ctlg"

columns_list = [
    {"name": "iss_cd",                            "data_type": "string"},
    {"name": "dmn_cd",                            "data_type": "string"},
    {"name": "stg_cd",                            "data_type": "string"},
    {"name": "pattern_cd",                        "data_type": "string"},
    {"name": "sev_cd",                            "data_type": "string"},
    {"name": "scp_cd",                            "data_type": "string"},
    {"name": "msg_tmplt_ln",                      "data_type": "string"},
    {"name": "param_nm_list_ln",                  "data_type": "string"},
    {"name": "act_flg",                           "data_type": "int"},
    {"name": "row_present_flg",                   "data_type": "int"},
    {"name": "dpm_vrs_dt",                        "data_type": "timestamp"},
    {"name": "dpm_ingestion_dt",                  "data_type": "timestamp"},
    {"name": "sk_dp_lgl_reg_dpm_param_iss_ctlg",  "data_type": "bigint"},
    {"name": "ref_start_dt",                      "data_type": "timestamp"},
    {"name": "ref_end_dt",                        "data_type": "timestamp"},
    {"name": "ref_last_dt",                       "data_type": "timestamp"},
    {"name": "ctrl_start_dt",                     "data_type": "timestamp"},
    {"name": "ctrl_end_dt",                       "data_type": "timestamp"},
    {"name": "ctrl_updt_dt",                      "data_type": "timestamp"},
    {"name": "ctrl_state",                        "data_type": "int"}
]

partition_cols = []

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

##### **dp_lgl_reg_dpm_ref_src_datapoint_typ**

In [ ]:
guids = set()
source_table_name = "dp_lgl_reg_dpm_ref_src_datapoint_typ"

columns_list = [
    {"name": "src_tbl_id",     "data_type": "string"},
    {"name": "datapoint_typ",  "data_type": "string"}
]

partition_cols = []

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

##### **dp_lgl_reg_dpm_ref_tmplt_csl_perim**

In [ ]:
guids = set()
source_table_name = "dp_lgl_reg_dpm_ref_tmplt_csl_perim"

columns_list = [
    {"name": "framework_cd",   "data_type": "string"},
    {"name": "tmplt_cd",       "data_type": "string"},
    {"name": "csl_perim_typ",  "data_type": "string"}
]

partition_cols = []

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

##### **dp_lgl_reg_dpm_ref_tmplt_ctry**

In [ ]:
guids = set()
source_table_name = "dp_lgl_reg_dpm_ref_tmplt_ctry"

columns_list = [
    {"name": "framework_cd",  "data_type": "string"},
    {"name": "tmplt_cd",  "data_type": "string"},
    {"name": "ctry_flg",  "data_type": "int"}
]

partition_cols = []

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

##### **dp_lgl_reg_dpm_param_vld**

In [ ]:
guids = set()
source_table_name = "dp_lgl_reg_dpm_param_vld"

columns_list = [
    {"name": "vld_id",          "data_type": "string"},
    {"name": "framework_cd",    "data_type": "string"},
    {"name": "param_snpsh_dt",  "data_type": "timestamp"},
    {"name": "vld_stts",        "data_type": "string"},
    {"name": "err_nr",          "data_type": "bigint"},
    {"name": "crtn_dt",         "data_type": "timestamp"}
]

partition_cols = []

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

##### **dp_lgl_reg_dpm_param_load**

In [ ]:
guids = set()
source_table_name = "dp_lgl_reg_dpm_param_load"

columns_list = [
    {"name": "src_file_nm",        "data_type": "string"},
    {"name": "framework_cd",       "data_type": "string"},
    {"name": "datapoint_load_id",  "data_type": "string"},
    {"name": "fltr_load_id",       "data_type": "string"},
    {"name": "ctlg_load_id",       "data_type": "string"},
    {"name": "src_load_id",        "data_type": "string"},
    {"name": "load_usr_id",        "data_type": "string"},
    {"name": "load_sts_cd",        "data_type": "string"},
    {"name": "vld_id",             "data_type": "string"},
    {"name": "param_snpsh_dt",     "data_type": "timestamp"},
    {"name": "crtn_dt",            "data_type": "timestamp"}
]

partition_cols = []

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

##### **dp_lgl_reg_dpm_exec**

In [ ]:
guids = set()
source_table_name = "dp_lgl_reg_dpm_exec"

columns_list = [
    {"name": "exec_id",            "data_type": "string"},
    {"name": "ref_dt",             "data_type": "timestamp"},
    {"name": "framework_list_ln",  "data_type": "string"},
    {"name": "param_snpsh_dt",     "data_type": "timestamp"},
    {"name": "exec_mode_cd",       "data_type": "string"},
    {"name": "exec_stts",          "data_type": "string"},
    {"name": "warning_nr",         "data_type": "bigint"},
    {"name": "err_nr",             "data_type": "bigint"},
    {"name": "start_dt",           "data_type": "timestamp"},
    {"name": "end_dt",             "data_type": "timestamp"},
    {"name": "msg_ln",             "data_type": "string"},
    {"name": "req_id",             "data_type": "string"}
]

partition_cols = []

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

##### **dp_lgl_reg_dpm_exec_step**

In [ ]:
guids = set()
source_table_name = "dp_lgl_reg_dpm_exec_step"

columns_list = [
    {"name": "exec_id",    "data_type": "string"},
    {"name": "step_nm",    "data_type": "string"},
    {"name": "grp_id",     "data_type": "string"},
    {"name": "step_stts",  "data_type": "string"},
    {"name": "row_nr",     "data_type": "bigint"},
    {"name": "msg_ln",     "data_type": "string"},
    {"name": "start_dt",   "data_type": "timestamp"},
    {"name": "end_dt",     "data_type": "timestamp"}
]

partition_cols = []

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

##### **dp_lgl_reg_dpm_iss_log**

In [ ]:
guids = set()
source_table_name = "dp_lgl_reg_dpm_iss_log"

columns_list = [
    {"name": "iss_id",          "data_type": "string"},
    {"name": "exec_id",         "data_type": "string"},
    {"name": "vld_id",          "data_type": "string"},
    {"name": "phs_cd",          "data_type": "string"},
    {"name": "param_snpsh_dt",  "data_type": "timestamp"},
    {"name": "ref_dt",          "data_type": "timestamp"},
    {"name": "iss_cd",          "data_type": "string"},
    {"name": "sev_cd",          "data_type": "string"},
    {"name": "framework_cd",    "data_type": "string"},
    {"name": "src_tbl_id",      "data_type": "string"},
    {"name": "datapoint_id",    "data_type": "string"},
    {"name": "var_id",          "data_type": "string"},
    {"name": "grp_id",          "data_type": "string"},
    {"name": "afct_nr",         "data_type": "bigint"},
    {"name": "param_json_ln",   "data_type": "string"},
    {"name": "msg_ln",          "data_type": "string"},
    {"name": "raw_err_cla_nm",  "data_type": "string"},
    {"name": "raw_msg_ln",      "data_type": "string"},
    {"name": "crtn_dt",         "data_type": "timestamp"}
]

partition_cols = []

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

##### **dp_lgl_reg_dpm_result_var_val**

In [ ]:
guids = set()
source_table_name = "dp_lgl_reg_dpm_result_var_val"

columns_list = [
    {"name": "exec_id",              "data_type": "string"},
    {"name": "ref_dt",               "data_type": "timestamp"},
    {"name": "framework_cd",         "data_type": "string"},
    {"name": "datapoint_id",         "data_type": "string"},
    {"name": "var_id",               "data_type": "string"},
    {"name": "var_cd",               "data_type": "string"},
    {"name": "var_cl_typ",           "data_type": "string"},
    {"name": "src_tbl_id",           "data_type": "string"},
    {"name": "per_cd",               "data_type": "string"},
    {"name": "per_frq_cd",           "data_type": "string"},
    {"name": "grain_id",             "data_type": "string"},
    {"name": "key_hsh_cd",           "data_type": "string"},
    {"name": "key_json_ln",          "data_type": "string"},
    {"name": "val_amnt",             "data_type": "decimal(38,10)"},
    {"name": "contributing_row_nr",  "data_type": "bigint"},
    {"name": "param_snpsh_dt",       "data_type": "timestamp"},
    {"name": "crtn_dt",              "data_type": "timestamp"}
]

partition_cols = []

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

##### **dp_lgl_reg_dpm_result_datapoint_val**

In [ ]:
guids = set()
source_table_name = "dp_lgl_reg_dpm_result_datapoint_val"

columns_list = [
    {"name": "exec_id",              "data_type": "string"},
    {"name": "ref_dt",               "data_type": "timestamp"},
    {"name": "per_cd",               "data_type": "string"},
    {"name": "per_frq_cd",           "data_type": "string"},
    {"name": "framework_cd",         "data_type": "string"},
    {"name": "tmplt_cd",             "data_type": "string"},
    {"name": "row_cd",               "data_type": "string"},
    {"name": "column_cd",            "data_type": "string"},
    {"name": "datapoint_id",         "data_type": "string"},
    {"name": "grain_id",             "data_type": "string"},
    {"name": "key_hsh_cd",           "data_type": "string"},
    {"name": "key_json_ln",          "data_type": "string"},
    {"name": "param_snpsh_dt",       "data_type": "timestamp"},
    {"name": "val_amnt",             "data_type": "decimal(38,10)"},
    {"name": "contributing_row_nr",  "data_type": "bigint"},
    {"name": "csl_ent_cd",           "data_type": "string"},
    {"name": "datapoint_typ",        "data_type": "string"},
    {"name": "csl_perim_typ",        "data_type": "string"},
    {"name": "csl_view_cd",          "data_type": "string"},
    {"name": "ctry_cd",              "data_type": "string"},
    {"name": "crtn_dt",              "data_type": "timestamp"}
]

partition_cols = []

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

##### **dp_lgl_reg_dpm_failed_var_key**

In [ ]:
guids = set()
source_table_name = "dp_lgl_reg_dpm_failed_var_key"

columns_list = [
    {"name": "exec_id",       "data_type": "string"},
    {"name": "framework_cd",  "data_type": "string"},
    {"name": "datapoint_id",  "data_type": "string"},
    {"name": "var_id",        "data_type": "string"},
    {"name": "key_hsh_cd",    "data_type": "string"},
    {"name": "iss_cd",        "data_type": "string"},
    {"name": "crtn_dt",       "data_type": "timestamp"}
]

partition_cols = []

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

##### **dp_lgl_reg_dpm_lineage_var**

In [ ]:
guids = set()
source_table_name = "dp_lgl_reg_dpm_lineage_var"

columns_list = [
    {"name": "exec_id",         "data_type": "string"},
    {"name": "ref_dt",          "data_type": "timestamp"},
    {"name": "framework_cd",    "data_type": "string"},
    {"name": "datapoint_id",    "data_type": "string"},
    {"name": "var_id",          "data_type": "string"},
    {"name": "var_cd",          "data_type": "string"},
    {"name": "src_tbl_id",      "data_type": "string"},
    {"name": "agg_fcn_cd",      "data_type": "string"},
    {"name": "metric_fmla_ln",  "data_type": "string"},
    {"name": "having_ln",       "data_type": "string"},
    {"name": "src_sql_ln",      "data_type": "string"},
    {"name": "grain_id",        "data_type": "string"},
    {"name": "per_cd",          "data_type": "string"},
    {"name": "param_snpsh_dt",  "data_type": "timestamp"},
    {"name": "crtn_dt",         "data_type": "timestamp"}
]

partition_cols = []

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

##### **dp_lgl_reg_dpm_lineage_datapoint**

In [ ]:
guids = set()
source_table_name = "dp_lgl_reg_dpm_lineage_datapoint"

columns_list = [
    {"name": "exec_id",           "data_type": "string"},
    {"name": "ref_dt",            "data_type": "timestamp"},
    {"name": "per_cd",            "data_type": "string"},
    {"name": "per_frq_cd",        "data_type": "string"},
    {"name": "framework_cd",      "data_type": "string"},
    {"name": "datapoint_id",      "data_type": "string"},
    {"name": "tmplt_cd",          "data_type": "string"},
    {"name": "row_cd",            "data_type": "string"},
    {"name": "column_cd",         "data_type": "string"},
    {"name": "datapoint_cl_typ",  "data_type": "string"},
    {"name": "fmla_ln",           "data_type": "string"},
    {"name": "var_map_json_ln",   "data_type": "string"},
    {"name": "csl_flg",           "data_type": "int"},
    {"name": "datapoint_sql_ln",  "data_type": "string"},
    {"name": "param_snpsh_dt",    "data_type": "timestamp"},
    {"name": "crtn_dt",           "data_type": "timestamp"}
]

partition_cols = []

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

##### **dp_lgl_reg_dpm_datapoint_def**

In [ ]:
guids = set()
source_table_name = "dp_lgl_reg_dpm_datapoint_def"

columns_list = [
    {"name": "exec_id",         "data_type": "string"},
    {"name": "ref_dt",          "data_type": "timestamp"},
    {"name": "framework_cd",    "data_type": "string"},
    {"name": "datapoint_id",    "data_type": "string"},
    {"name": "tmplt_cd",        "data_type": "string"},
    {"name": "row_cd",          "data_type": "string"},
    {"name": "column_cd",       "data_type": "string"},
    {"name": "csl_ent_cd",      "data_type": "string"},
    {"name": "csl_view_cd",     "data_type": "string"},
    {"name": "csl_perim_typ",   "data_type": "string"},
    {"name": "datapoint_typ",   "data_type": "string"},
    {"name": "ctry_cd",         "data_type": "string"},
    {"name": "def_val_ln",      "data_type": "string"},
    {"name": "param_snpsh_dt",  "data_type": "timestamp"},
    {"name": "crtn_dt",         "data_type": "timestamp"}
]

partition_cols = []

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

##### **dp_lgl_reg_dpm_rpt_datapoint**

In [ ]:
guids = set()
source_table_name = "dp_lgl_reg_dpm_rpt_datapoint"

columns_list = [
    {"name": "exec_id",        "data_type": "string"},
    {"name": "ref_dt",         "data_type": "timestamp"},
    {"name": "framework_cd",   "data_type": "string"},
    {"name": "datapoint_id",   "data_type": "string"},
    {"name": "tmplt_cd",       "data_type": "string"},
    {"name": "row_cd",         "data_type": "string"},
    {"name": "column_cd",      "data_type": "string"},
    {"name": "csl_ent_cd",     "data_type": "string"},
    {"name": "csl_view_cd",    "data_type": "string"},
    {"name": "csl_perim_typ",  "data_type": "string"},
    {"name": "datapoint_typ",  "data_type": "string"},
    {"name": "ctry_cd",        "data_type": "string"},
    {"name": "rpt_val_ln",     "data_type": "string"},
    {"name": "pbh_dt",         "data_type": "timestamp"}
]

partition_cols = []

create_table(source_table_name, 'TABLE', columns_list, reset_list, partition_cols, guids)

#### **Engine Execution**
___

In [8]:
#Engine setup
engine = workerfactory.WorkerFactory(spark,mssparkutils,medallion_framework,verbose=False)
engine.send_finish_notification()

StatementMeta(, 7046c2f2-d03c-415d-aaf3-60a26991c97b, 12, Finished, Available, Finished, False)

2026-11-19 12:19:59,149 - WorkerFactory - send_finish_notification - ⚠️ WARNING - ⚠️ Environment 'FET' does not have Orchestration. Skipping 'AgentEvent.End' notification.)
